# ALQAC — Outcome Prediction không dùng public data

Pipeline này **không đọc public test, public labels, public verdict hoặc public
law annotations**.

```text
private raw case evidence
+ private selected law evidence
        ↓
Qwen3.5-9B primary judge
        ↓
unload model
        ↓
Aya Expanse 8B independent auditor
        ↓
deterministic claim-rule consensus
        ↓
submission.json
```

Không có KNN, few-shot public, calibration public hoặc learned classifier.

Nếu Aya không load được do chưa chấp nhận license, notebook có thể fallback sang
`Qwen/Qwen3-8B`. Để giữ tính độc lập giữa hai model family, nên dùng Aya.


## 0. Chuẩn bị model Aya

Trước lần chạy đầu:

1. Mở trang model `CohereLabs/aya-expanse-8b` trên Hugging Face.
2. Chấp nhận điều kiện truy cập.
3. Tạo Hugging Face token có quyền đọc.
4. Dán token vào cell cấu hình, hoặc đặt Colab Secret tên `HF_TOKEN`.

Aya chỉ được dùng làm independent auditor; Qwen3.5-9B là primary judge.


In [5]:
!pip -q install -U \
  "transformers @ git+https://github.com/huggingface/transformers.git@main" \
  accelerate \
  bitsandbytes \
  safetensors \
  einops \
  pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
from pathlib import Path
from getpass import getpass
import json
import os
import shutil
import sys

import pandas as pd
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn A100 GPU.")
print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.11.0+cu130
CUDA: True
GPU: NVIDIA A100-SXM4-80GB


## 1. Ghi module

Primary/auditor outputs được lưu riêng từng case để resume.


In [7]:
%%writefile /content/alqac_outcome_no_public.py

from __future__ import annotations

import gc
import json
import math
import os
import re
import shutil
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Optional, Sequence

import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForMultimodalLM,
    AutoProcessor,
    AutoTokenizer,
    BitsAndBytesConfig,
)


# =============================================================================
# Constants
# =============================================================================

LABELS = (
    "A_WIN",
    "PARTIAL_A_WIN",
    "PARTIAL_B_WIN",
    "B_WIN",
)

LABEL_INDEX = {label: index for index, label in enumerate(LABELS)}

LABEL_DEFINITIONS = {
    "A_WIN": (
        "Nguyên đơn A được chấp nhận về cơ bản toàn bộ hoặc phần cốt lõi của "
        "yêu cầu chính; B không thắng phản tố hoặc defense có trọng lượng tương "
        "đương."
    ),
    "PARTIAL_A_WIN": (
        "A và B đều đạt một phần, nhưng A nhận phần cứu trợ chính hoặc có trọng "
        "lượng lớn hơn; thường trách nhiệm của B được xác lập nhưng mức/phạm vi "
        "yêu cầu của A bị giảm."
    ),
    "PARTIAL_B_WIN": (
        "A chỉ nhận cứu trợ hạn chế hoặc thứ yếu, còn B bác được phần lớn yêu "
        "cầu chính hoặc thắng vấn đề/phản tố có trọng lượng lớn hơn."
    ),
    "B_WIN": (
        "Yêu cầu chính của A bị bác về cơ bản/toàn bộ, hoặc vị trí pháp lý chính "
        "của B được chấp nhận rõ rệt."
    ),
}

RESULT_VALUES = {
    "GRANTED",
    "PARTIALLY_GRANTED",
    "REJECTED",
    "UNCERTAIN",
}

CLAIM_TYPES = {
    "A_CLAIM",
    "B_COUNTERCLAIM",
    "B_DEFENSE",
    "PROCEDURAL",
}

IMPORTANCE_VALUES = {
    "CORE",
    "MAJOR",
    "SECONDARY",
}

IMPORTANCE_WEIGHT = {
    "CORE": 4.0,
    "MAJOR": 2.5,
    "SECONDARY": 1.0,
}

GRANT_FRACTION = {
    "GRANTED": 1.0,
    "PARTIALLY_GRANTED": 0.55,
    "REJECTED": 0.0,
    "UNCERTAIN": 0.5,
}

SOURCE_LABELS = {
    "plaintiff": "NGUYÊN ĐƠN",
    "defendant": "BỊ ĐƠN",
    "witness": "NHÂN CHỨNG/NGƯỜI CHỨNG KIẾN",
    "prosecutor": "VIỆN KIỂM SÁT",
    "unknown": "NGUỒN KHÁC/CHƯA XÁC ĐỊNH",
}

LAW_TITLES: dict[str, str] = {
    "47/2010/QH12": "Luật Các tổ chức tín dụng 2010",
    "26/2008/QH12": "Luật Thi hành án dân sự 2008",
    "36/2005/QH11": "Luật Thương mại 2005",
    "43/2014/NĐ-CP": "Nghị định 43/2014/NĐ-CP hướng dẫn Luật Đất đai",
    "326/2016/UBTVQH14": "Nghị quyết 326/2016/UBTVQH14 về án phí và lệ phí Tòa án",
    "11/2017/QH14": "Luật Trợ giúp pháp lý 2017",
    "92/2015/QH13": "Bộ luật Tố tụng dân sự 2015",
    "39/2016/TT-NHNN": "Thông tư 39/2016/TT-NHNN về hoạt động cho vay",
    "91/2015/QH13": "Bộ luật Dân sự 2015",
    "52/2014/QH13": "Luật Hôn nhân và gia đình 2014",
    "59/2020/QH14": "Luật Doanh nghiệp 2020",
    "53/2014/QH13": "Luật Phá sản 2014",
    "45/2013/QH13": "Luật Đất đai 2013",
    "46/2010/QH12": "Luật Ngân hàng Nhà nước Việt Nam 2010",
}

PREDICTION_MARKERS = (
    "agent cần dự đoán",
    "agent dự đoán",
    "hãy dự đoán",
    "cần dự đoán",
    "dự đoán ai",
    "ai được chấp nhận yêu cầu",
    "ai thắng kiện",
    "nguyên đơn thắng kiện hay bị đơn",
    "nguyên đơn hay bị đơn thắng",
)

OUTCOME_CLUE_PATTERNS = (
    r"\bđề nghị\b.{0,100}\bchấp nhận\b",
    r"\bđề nghị\b.{0,100}\bbác\b",
    r"\bchấp nhận toàn bộ\b",
    r"\bchấp nhận một phần\b",
    r"\bbác toàn bộ\b",
    r"\bbác (?:một phần|yêu cầu)\b",
    r"\bkhông chấp nhận\b",
    r"\bgiữ nguyên bản án\b",
    r"\bsửa (?:một phần )?bản án\b",
    r"\bhủy bản án\b",
    r"\bđình chỉ xét xử\b",
    r"\brút kháng nghị\b",
    r"\bchấp nhận phản tố\b",
    r"\bbác phản tố\b",
)


# =============================================================================
# Configuration and data classes
# =============================================================================

@dataclass
class NoPublicOutcomeConfig:
    primary_model_id: str = "Qwen/Qwen3.5-9B"
    auditor_model_id: str = "CohereLabs/aya-expanse-8b"
    auditor_fallback_model_id: str = "Qwen/Qwen3-8B"

    primary_quant_mode: str = "bf16"  # bf16 | bnb4 | bnb8
    auditor_quant_mode: str = "bf16"
    allow_auditor_fallback: bool = True
    hf_token: Optional[str] = None

    deterministic_generation: bool = True
    qwen_enable_thinking: bool = False

    # Packet must also fit Aya's shorter context.
    max_chunks: int = 24
    max_words_per_chunk: int = 125
    max_total_evidence_words: int = 1900
    max_law_articles: int = 16
    max_words_per_law: int = 150
    max_total_law_words: int = 900
    max_hint_words: int = 230
    max_outcome_clues: int = 10

    primary_max_new_tokens: int = 950
    auditor_max_new_tokens: int = 950

    # No learned/public calibration. Fixed transparent consensus.
    direct_primary_weight: float = 0.32
    claim_primary_weight: float = 0.22
    direct_auditor_weight: float = 0.27
    claim_auditor_weight: float = 0.19

    # Reliability adjustment.
    min_valid_claims_for_full_weight: int = 2
    evidence_citation_bonus: float = 0.08
    law_citation_bonus: float = 0.05
    disagreement_claim_rule_bonus: float = 0.08

    expected_private_cases: int = 60


@dataclass
class LawArticle:
    aid: int
    law_id: str
    law_title: str
    article_number: int
    text: str


@dataclass
class CaseBundle:
    case_id: str
    case_query: str
    clean_query: str
    raw_chunks: list[dict[str, Any]]
    selected_chunk_ids: list[str]
    selected_law_aids: list[int]
    hints: dict[str, Any]
    packet_text: str = ""
    valid_chunk_ids: set[str] = field(default_factory=set)
    valid_law_aids: set[int] = field(default_factory=set)


# =============================================================================
# Generic helpers
# =============================================================================

def normalize_text(text: Any) -> str:
    value = unicodedata.normalize("NFC", str(text or "")).lower()
    value = re.sub(r"[^\w\s/.-]", " ", value, flags=re.UNICODE)
    return " ".join(value.split())


def clean_case_query(case_query: str) -> str:
    text = " ".join(str(case_query or "").split())
    lowered = text.lower()
    positions = [
        lowered.find(marker)
        for marker in PREDICTION_MARKERS
        if lowered.find(marker) >= 0
    ]
    if positions:
        text = text[:min(positions)]
    return text.strip(" .?;:")


def truncate_words(text: Any, max_words: int) -> str:
    words = str(text or "").split()
    if len(words) <= max_words:
        return " ".join(words)
    return " ".join(words[:max_words]) + " …"


def dedupe_preserve(values: Iterable[Any]) -> list[Any]:
    seen = set()
    output = []
    for value in values:
        marker = (
            json.dumps(value, ensure_ascii=False, sort_keys=True)
            if isinstance(value, (dict, list))
            else value
        )
        if marker in seen:
            continue
        seen.add(marker)
        output.append(value)
    return output


def atomic_write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(path)


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8-sig"))


def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        result = float(value)
    except (TypeError, ValueError):
        return default
    if not math.isfinite(result):
        return default
    return result


def normalize_probabilities(
    raw: Any,
    predicted_label: Optional[str] = None,
) -> dict[str, float]:
    output = {label: 0.0 for label in LABELS}
    if isinstance(raw, dict):
        for key, value in raw.items():
            key_text = str(key).upper().strip()
            if key_text in output:
                output[key_text] = max(0.0, safe_float(value))

    total = sum(output.values())
    if total <= 0:
        if predicted_label in output:
            output = {label: 0.10 for label in LABELS}
            output[predicted_label] = 0.70
        else:
            output = {label: 0.25 for label in LABELS}
    else:
        output = {
            label: value / total
            for label, value in output.items()
        }
    return output


def top_label(probabilities: dict[str, float]) -> str:
    return max(
        LABELS,
        key=lambda label: (
            probabilities.get(label, 0.0),
            -LABEL_INDEX[label],
        ),
    )


def probability_margin(probabilities: dict[str, float]) -> float:
    ordered = sorted(probabilities.values(), reverse=True)
    if len(ordered) < 2:
        return 1.0
    return float(ordered[0] - ordered[1])


def numeric_case_key(case_id: str) -> tuple[int, str]:
    match = re.search(r"\d+", str(case_id))
    return (
        int(match.group()) if match else 10**9,
        str(case_id),
    )


# =============================================================================
# Corpus and private data loading
# =============================================================================

def load_law_corpus(
    path: Path,
) -> dict[int, LawArticle]:
    payload = read_json(path)
    output: dict[int, LawArticle] = {}

    for document in payload:
        law_id = str(document["law_id"])
        law_title = LAW_TITLES.get(law_id, law_id)
        for article_number, article in enumerate(
            document.get("content", []),
            start=1,
        ):
            aid = int(article["aid"])
            output[aid] = LawArticle(
                aid=aid,
                law_id=law_id,
                law_title=law_title,
                article_number=article_number,
                text=str(
                    article.get("content_Article", "")
                ).strip(),
            )

    if not output:
        raise ValueError("Corpus luật rỗng.")
    return output


def find_case_files(
    directory: Path,
    required_fragment: str,
) -> dict[str, Path]:
    output: dict[str, Path] = {}
    for path in directory.rglob("*.json"):
        lower = path.name.lower()
        if required_fragment not in lower:
            continue
        match = re.search(r"(case_\d+)", lower)
        if not match:
            continue
        case_id = match.group(1)
        current = output.get(case_id)
        if current is None or len(str(path)) < len(str(current)):
            output[case_id] = path
    return output


def load_law_final_rows(
    path: Path,
) -> dict[str, dict[str, Any]]:
    payload = read_json(path)
    if not isinstance(payload, list):
        raise ValueError(
            "private_case_and_law_evidence.json phải là JSON array."
        )

    output = {}
    for row in payload:
        case_id = str(row.get("case_id", ""))
        if not case_id:
            continue
        if case_id in output:
            raise ValueError(
                f"Trùng case_id trong law output: {case_id}"
            )
        output[case_id] = row
    return output


def choose_chunk_source(
    chunk: dict[str, Any],
) -> str:
    observed = chunk.get("observed_source_types", [])
    if not isinstance(observed, list):
        observed = []
    for source_type in (
        "plaintiff",
        "defendant",
        "witness",
        "prosecutor",
    ):
        if source_type in observed:
            return source_type

    attributions = chunk.get("source_attributions", [])
    if isinstance(attributions, list):
        for item in attributions:
            source_type = str(
                item.get("source_type", "")
            ).lower()
            if source_type in SOURCE_LABELS:
                return source_type
    return "unknown"


def load_private_cases(
    private_test_path: Path,
    case_retrieval_dir: Path,
    private_case_and_law_path: Path,
) -> tuple[list[CaseBundle], list[dict[str, Any]]]:
    original_cases = read_json(private_test_path)
    evidence_files = find_case_files(
        case_retrieval_dir,
        "evidence_result",
    )
    handoff_files = find_case_files(
        case_retrieval_dir,
        "law_handoff",
    )
    law_rows = load_law_final_rows(
        private_case_and_law_path
    )

    cases = []
    for original in original_cases:
        case_id = str(original["case_id"])
        evidence_path = evidence_files.get(case_id)
        if evidence_path is None:
            raise FileNotFoundError(
                f"Thiếu evidence_result cho {case_id}"
            )

        result = read_json(evidence_path)
        handoff = (
            read_json(handoff_files[case_id])
            if case_id in handoff_files
            else result.get("handoff_for_later_stages", {})
        )
        law_row = law_rows.get(case_id)
        if law_row is None:
            raise ValueError(
                f"Thiếu law retrieval output cho {case_id}"
            )

        chunks = result.get("all_unique_chunks", [])
        if not isinstance(chunks, list):
            chunks = []

        selected_ids = [
            str(value)
            for value in law_row.get(
                "case_evidence",
                handoff.get("case_evidence", []),
            )
            if str(value)
        ]

        selected_law_aids = []
        for value in law_row.get("law_evidence", []):
            if isinstance(value, dict):
                value = value.get("aid")
            try:
                selected_law_aids.append(int(value))
            except (TypeError, ValueError):
                continue

        hints = {
            "atomic_facts": handoff.get(
                "atomic_facts",
                result.get("atomic_facts", []),
            ),
            "evidence_topics": handoff.get(
                "evidence_topics",
                result.get("evidence_topics", []),
            ),
            "unresolved_factual_gaps": handoff.get(
                "unresolved_factual_gaps",
                result.get("unresolved_factual_gaps", []),
            ),
            "source_coverage": handoff.get(
                "source_coverage",
                result.get("source_coverage", {}),
            ),
        }

        cases.append(CaseBundle(
            case_id=case_id,
            case_query=str(
                result.get(
                    "case_query",
                    original.get("case_query", ""),
                )
            ),
            clean_query=str(
                result.get(
                    "clean_query",
                    clean_case_query(
                        original.get("case_query", "")
                    ),
                )
            ),
            raw_chunks=chunks,
            selected_chunk_ids=dedupe_preserve(selected_ids),
            selected_law_aids=dedupe_preserve(
                selected_law_aids
            ),
            hints=hints,
        ))

    return cases, original_cases


# =============================================================================
# Packet construction
# =============================================================================

def extract_outcome_clues(
    chunks: Sequence[dict[str, Any]],
    max_clues: int,
) -> list[dict[str, str]]:
    output = []
    seen = set()

    for chunk in chunks:
        text = " ".join(str(chunk.get("text", "")).split())
        if not text:
            continue
        sentences = re.split(
            r"(?<=[.!?;:])\s+|\n+",
            text,
        )
        for sentence in sentences:
            normalized = normalize_text(sentence)
            if not normalized:
                continue
            if not any(
                re.search(pattern, normalized)
                for pattern in OUTCOME_CLUE_PATTERNS
            ):
                continue
            clipped = truncate_words(sentence, 85)
            marker = normalize_text(clipped)
            if marker in seen:
                continue
            seen.add(marker)
            output.append({
                "chunk_id": str(
                    chunk.get("chunk_id", "")
                ),
                "source": choose_chunk_source(chunk),
                "text": clipped,
            })
            if len(output) >= max_clues:
                return output
    return output


def priority_chunks(
    case: CaseBundle,
    config: NoPublicOutcomeConfig,
) -> list[dict[str, Any]]:
    by_id = {
        str(chunk.get("chunk_id", "")): chunk
        for chunk in case.raw_chunks
        if chunk.get("chunk_id")
    }

    ordered = [
        by_id[chunk_id]
        for chunk_id in case.selected_chunk_ids
        if chunk_id in by_id
    ]

    source_coverage = case.hints.get(
        "source_coverage",
        {},
    )
    if isinstance(source_coverage, dict):
        for source_type in (
            "plaintiff",
            "defendant",
            "witness",
            "prosecutor",
        ):
            item = source_coverage.get(source_type, {})
            if not isinstance(item, dict):
                continue
            for chunk_id in item.get("evidence_ids", []):
                chunk_id = str(chunk_id)
                if (
                    chunk_id in by_id
                    and by_id[chunk_id] not in ordered
                ):
                    ordered.append(by_id[chunk_id])

    for chunk in case.raw_chunks:
        if chunk not in ordered:
            ordered.append(chunk)

    buckets = defaultdict(list)
    for chunk in ordered:
        buckets[choose_chunk_source(chunk)].append(chunk)

    output = []
    source_order = (
        "plaintiff",
        "defendant",
        "witness",
        "prosecutor",
        "unknown",
    )
    while len(output) < config.max_chunks:
        added = False
        for source_type in source_order:
            if buckets[source_type]:
                output.append(
                    buckets[source_type].pop(0)
                )
                added = True
                if len(output) >= config.max_chunks:
                    break
        if not added:
            break
    return output


def build_hint_text(
    hints: dict[str, Any],
    max_words: int,
) -> str:
    parts = []

    facts = hints.get("atomic_facts", [])
    if isinstance(facts, list):
        rows = []
        for item in facts[:10]:
            value = (
                item.get("fact", "")
                if isinstance(item, dict)
                else item
            )
            value = " ".join(str(value).split())
            if value:
                rows.append(f"- {value}")
        if rows:
            parts.append(
                "FACT CANDIDATES DO MODEL RETRIEVAL SINH "
                "(chỉ là hints):\n"
                + "\n".join(rows)
            )

    topics = hints.get("evidence_topics", [])
    if isinstance(topics, list):
        rows = []
        for item in topics[:8]:
            value = (
                item.get("title")
                or item.get("description")
                or ""
            ) if isinstance(item, dict) else item
            value = " ".join(str(value).split())
            if value:
                rows.append(f"- {value}")
        if rows:
            parts.append(
                "TOPIC HINTS:\n" + "\n".join(rows)
            )

    gaps = hints.get(
        "unresolved_factual_gaps",
        [],
    )
    if isinstance(gaps, list):
        rows = []
        for item in gaps[:8]:
            value = (
                item.get("description")
                or item.get("gap")
                or ""
            ) if isinstance(item, dict) else item
            value = " ".join(str(value).split())
            if value:
                rows.append(f"- {value}")
        if rows:
            parts.append(
                "ĐIỂM CHƯA CHẮC CHẮN:\n"
                + "\n".join(rows)
            )

    return truncate_words(
        "\n\n".join(parts),
        max_words,
    )


def build_law_block(
    aids: Sequence[int],
    article_by_aid: dict[int, LawArticle],
    config: NoPublicOutcomeConfig,
) -> str:
    blocks = []
    used_words = 0

    for aid in aids[:config.max_law_articles]:
        article = article_by_aid.get(int(aid))
        if article is None:
            continue
        remaining = config.max_total_law_words - used_words
        if remaining <= 0:
            break
        max_words = min(
            config.max_words_per_law,
            remaining,
        )
        text = truncate_words(article.text, max_words)
        used_words += len(text.split())
        blocks.append(
            f"[aid={article.aid}; law_id={article.law_id}; "
            f"{article.law_title}; Điều {article.article_number}]\n"
            f"{text}"
        )

    return "\n\n".join(blocks)


def build_case_packet(
    case: CaseBundle,
    article_by_aid: dict[int, LawArticle],
    config: NoPublicOutcomeConfig,
) -> None:
    chunks = priority_chunks(case, config)
    selected_set = set(case.selected_chunk_ids)

    evidence_blocks = []
    used_words = 0
    for chunk in chunks:
        remaining = (
            config.max_total_evidence_words - used_words
        )
        if remaining <= 0:
            break
        max_words = min(
            config.max_words_per_chunk,
            remaining,
        )
        text = truncate_words(
            chunk.get("text", ""),
            max_words,
        )
        if not text:
            continue
        used_words += len(text.split())
        source = choose_chunk_source(chunk)
        status = (
            "SELECTED"
            if str(chunk.get("chunk_id")) in selected_set
            else "SUPPLEMENTAL"
        )
        evidence_blocks.append(
            f"[{chunk.get('chunk_id')} | "
            f"{SOURCE_LABELS[source]} | {status}]\n{text}"
        )

    clues = extract_outcome_clues(
        chunks,
        config.max_outcome_clues,
    )
    clue_block = "\n".join(
        f"- [{item['chunk_id']} | "
        f"{SOURCE_LABELS.get(item['source'], item['source'])}] "
        f"{item['text']}"
        for item in clues
    )

    hint_block = build_hint_text(
        case.hints,
        config.max_hint_words,
    )
    law_block = build_law_block(
        case.selected_law_aids,
        article_by_aid,
        config,
    )

    parts = [
        f"CASE ID: {case.case_id}",
        f"CASE QUERY GỐC:\n{case.case_query}",
        f"YÊU CẦU ĐÃ LÀM SẠCH:\n{case.clean_query}",
        "RAW CASE EVIDENCE:\n"
        + "\n\n".join(evidence_blocks),
    ]

    if clue_block:
        parts.append(
            "CÂU CÓ THỂ LIÊN QUAN ĐẾN KẾT QUẢ:\n"
            "Đây có thể chỉ là quan điểm của một bên, Viện kiểm sát hoặc cấp "
            "xét xử trước; không mặc định là kết quả cuối.\n"
            + clue_block
        )

    if hint_block:
        parts.append(hint_block)

    parts.append(
        "SELECTED LAW EVIDENCE FROM CORPUS:\n"
        + (law_block or "(không có)")
    )

    case.packet_text = "\n\n".join(parts)
    case.valid_chunk_ids = {
        str(chunk.get("chunk_id"))
        for chunk in case.raw_chunks
        if chunk.get("chunk_id")
    }
    case.valid_law_aids = {
        int(aid)
        for aid in case.selected_law_aids
        if int(aid) in article_by_aid
    }


# =============================================================================
# Runtime base and JSON parsing
# =============================================================================

class JsonRuntimeBase:
    @staticmethod
    def extract_json_object(text: str) -> dict[str, Any]:
        cleaned = str(text or "").strip()
        cleaned = re.sub(
            r"^```(?:json)?\s*",
            "",
            cleaned,
            flags=re.IGNORECASE,
        )
        cleaned = re.sub(r"\s*```$", "", cleaned)

        decoder = json.JSONDecoder()
        for index, char in enumerate(cleaned):
            if char != "{":
                continue
            try:
                value, _ = decoder.raw_decode(
                    cleaned[index:]
                )
                if isinstance(value, dict):
                    return value
            except json.JSONDecodeError:
                continue
        raise ValueError(
            "Không parse được JSON:\n"
            + cleaned[:1800]
        )


class QwenPrimaryRuntime(JsonRuntimeBase):
    def __init__(
        self,
        config: NoPublicOutcomeConfig,
    ):
        self.config = config
        self.processor: Optional[Any] = None
        self.model: Optional[Any] = None

    def load(self) -> None:
        if not torch.cuda.is_available():
            raise RuntimeError("Hãy chọn A100 GPU.")

        torch.backends.cuda.matmul.allow_tf32 = True
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.allow_tf32 = True

        self.processor = AutoProcessor.from_pretrained(
            self.config.primary_model_id,
            trust_remote_code=False,
        )

        kwargs: dict[str, Any] = {
            "device_map": "auto",
            "torch_dtype": torch.bfloat16,
            "low_cpu_mem_usage": True,
            "trust_remote_code": False,
        }
        mode = self.config.primary_quant_mode.lower()
        if mode == "bnb4":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=False,
            )
        elif mode == "bnb8":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_8bit=True,
            )
        elif mode != "bf16":
            raise ValueError(
                "primary_quant_mode không hợp lệ."
            )

        self.model = AutoModelForMultimodalLM.from_pretrained(
            self.config.primary_model_id,
            **kwargs,
        )
        self.model.eval()
        print(
            "Loaded primary:",
            self.config.primary_model_id,
        )

    def unload(self) -> None:
        self.processor = None
        self.model = None
        gc.collect()
        torch.cuda.empty_cache()

    def generate_json(
        self,
        system: str,
        user: str,
        max_new_tokens: int,
    ) -> dict[str, Any]:
        if self.model is None or self.processor is None:
            raise RuntimeError("Primary runtime chưa load.")

        messages = [
            {
                "role": "system",
                "content": [{
                    "type": "text",
                    "text": system,
                }],
            },
            {
                "role": "user",
                "content": [{
                    "type": "text",
                    "text": user,
                }],
            },
        ]
        common = {
            "add_generation_prompt": True,
            "tokenize": True,
            "return_dict": True,
            "return_tensors": "pt",
        }
        try:
            inputs = self.processor.apply_chat_template(
                messages,
                enable_thinking=(
                    self.config.qwen_enable_thinking
                ),
                **common,
            )
        except TypeError:
            inputs = self.processor.apply_chat_template(
                messages,
                chat_template_kwargs={
                    "enable_thinking": (
                        self.config.qwen_enable_thinking
                    ),
                },
                **common,
            )
        inputs = inputs.to(self.model.device)
        input_length = inputs["input_ids"].shape[-1]

        generation_kwargs: dict[str, Any] = {
            "max_new_tokens": max_new_tokens,
            "do_sample": not (
                self.config.deterministic_generation
            ),
            "pad_token_id": (
                self.processor.tokenizer.eos_token_id
            ),
            "eos_token_id": (
                self.processor.tokenizer.eos_token_id
            ),
            "use_cache": True,
        }
        if not self.config.deterministic_generation:
            generation_kwargs.update({
                "temperature": 0.6,
                "top_p": 0.85,
            })

        with torch.inference_mode():
            generated = self.model.generate(
                **inputs,
                **generation_kwargs,
            )
        new_tokens = generated[0, input_length:]
        text = self.processor.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()
        return self.extract_json_object(text)


class TextAuditorRuntime(JsonRuntimeBase):
    def __init__(
        self,
        config: NoPublicOutcomeConfig,
    ):
        self.config = config
        self.model_id = config.auditor_model_id
        self.tokenizer: Optional[Any] = None
        self.model: Optional[Any] = None
        self.using_fallback = False

    def _load_model(self, model_id: str) -> None:
        token = self.config.hf_token or os.getenv("HF_TOKEN")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            token=token,
            trust_remote_code=False,
        )

        kwargs: dict[str, Any] = {
            "device_map": "auto",
            "torch_dtype": torch.bfloat16,
            "low_cpu_mem_usage": True,
            "trust_remote_code": False,
            "token": token,
        }
        mode = self.config.auditor_quant_mode.lower()
        if mode == "bnb4":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=False,
            )
        elif mode == "bnb8":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_8bit=True,
            )
        elif mode != "bf16":
            raise ValueError(
                "auditor_quant_mode không hợp lệ."
            )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            **kwargs,
        )
        self.model.eval()
        self.model_id = model_id

    def load(self) -> None:
        try:
            self._load_model(
                self.config.auditor_model_id
            )
        except Exception as exc:
            if not self.config.allow_auditor_fallback:
                raise
            print(
                "Không load được Aya auditor:",
                type(exc).__name__,
                exc,
            )
            print(
                "Fallback auditor:",
                self.config.auditor_fallback_model_id,
            )
            self.tokenizer = None
            self.model = None
            gc.collect()
            torch.cuda.empty_cache()
            self.using_fallback = True
            self._load_model(
                self.config.auditor_fallback_model_id
            )

        print(
            "Loaded auditor:",
            self.model_id,
            "fallback=",
            self.using_fallback,
        )

    def unload(self) -> None:
        self.tokenizer = None
        self.model = None
        gc.collect()
        torch.cuda.empty_cache()

    def generate_json(
        self,
        system: str,
        user: str,
        max_new_tokens: int,
    ) -> dict[str, Any]:
        if self.model is None or self.tokenizer is None:
            raise RuntimeError("Auditor runtime chưa load.")

        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        template_kwargs = {
            "tokenize": True,
            "add_generation_prompt": True,
            "return_tensors": "pt",
        }

        # Qwen3 fallback supports enable_thinking; Aya ignores it.
        try:
            input_ids = self.tokenizer.apply_chat_template(
                messages,
                enable_thinking=False,
                **template_kwargs,
            )
        except TypeError:
            input_ids = self.tokenizer.apply_chat_template(
                messages,
                **template_kwargs,
            )

        input_ids = input_ids.to(self.model.device)
        input_length = input_ids.shape[-1]

        generation_kwargs: dict[str, Any] = {
            "max_new_tokens": max_new_tokens,
            "do_sample": not (
                self.config.deterministic_generation
            ),
            "pad_token_id": (
                self.tokenizer.eos_token_id
            ),
            "eos_token_id": (
                self.tokenizer.eos_token_id
            ),
            "use_cache": True,
        }
        if not self.config.deterministic_generation:
            generation_kwargs.update({
                "temperature": 0.55,
                "top_p": 0.85,
            })

        with torch.inference_mode():
            generated = self.model.generate(
                input_ids,
                **generation_kwargs,
            )
        new_tokens = generated[0, input_length:]
        text = self.tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()
        return self.extract_json_object(text)


# =============================================================================
# Prompts and structured prediction
# =============================================================================

def label_definition_text() -> str:
    return "\n".join(
        f"- {label}: {LABEL_DEFINITIONS[label]}"
        for label in LABELS
    )


def prediction_schema() -> str:
    return """
Schema:
{
  "prediction": "A_WIN|PARTIAL_A_WIN|PARTIAL_B_WIN|B_WIN",
  "probabilities": {
    "A_WIN": 0.0,
    "PARTIAL_A_WIN": 0.0,
    "PARTIAL_B_WIN": 0.0,
    "B_WIN": 0.0
  },
  "claims": [
    {
      "claim_id": "c1",
      "claim_type": "A_CLAIM|B_COUNTERCLAIM|B_DEFENSE|PROCEDURAL",
      "claim": "mô tả ngắn",
      "importance": "CORE|MAJOR|SECONDARY",
      "result": "GRANTED|PARTIALLY_GRANTED|REJECTED|UNCERTAIN",
      "case_evidence_ids": ["chunk_id"],
      "law_aids": [123],
      "short_reason": "tối đa 55 từ"
    }
  ],
  "confidence": "HIGH|MEDIUM|LOW",
  "decision_summary": "tối đa 120 từ"
}
Các xác suất phải không âm và có tổng xấp xỉ 1.
"""


PRIMARY_SYSTEM = """
Bạn là mô hình xét xử kết quả vụ án dân sự Việt Nam.

A luôn là NGUYÊN ĐƠN trong case query; B luôn là BỊ ĐƠN, kể cả khi B là người
kháng cáo. Chỉ được chọn một trong bốn nhãn:
A_WIN, PARTIAL_A_WIN, PARTIAL_B_WIN, B_WIN.

Quy tắc:
1. Raw case evidence và law evidence nguyên văn là căn cứ chính.
2. Facts/topics/gaps do pipeline trước sinh chỉ là hints.
3. Phải tách yêu cầu của A, phản tố/defense của B và vấn đề thủ tục.
4. Không đếm số yêu cầu đơn thuần; đánh giá trọng lượng cứu trợ cốt lõi.
5. Ý kiến Viện kiểm sát hoặc kết quả cấp dưới không tự động là kết quả cuối.
6. Mọi claim result nên dẫn chunk_id và aid có thật trong input.
7. Không thêm facts, luật hoặc kết quả không có trong dữ liệu.
8. Chỉ trả JSON.
"""


AUDITOR_SYSTEM = """
Bạn là người phản biện độc lập, không được xác nhận máy móc một mô hình khác.

Hãy tự đánh giá vụ án dân sự Việt Nam từ raw case evidence và law evidence:
- A là nguyên đơn, B là bị đơn;
- xác định yêu cầu cốt lõi, phản tố và defense;
- phân biệt thắng toàn bộ với thắng một phần;
- PARTIAL_A_WIN khi A nhận cứu trợ có trọng lượng lớn hơn;
- PARTIAL_B_WIN khi B thắng vấn đề trung tâm nhưng A vẫn nhận cứu trợ hạn chế;
- B_WIN khi yêu cầu chính của A bị bác về cơ bản;
- không coi hints do model trước sinh là sự thật;
- không coi ý kiến Viện kiểm sát/cấp dưới là kết quả cuối;
- chỉ trả JSON.
"""


def validate_claim(
    claim: dict[str, Any],
    case: CaseBundle,
) -> dict[str, Any]:
    claim_type = str(
        claim.get("claim_type", "")
    ).upper()
    if claim_type not in CLAIM_TYPES:
        claim_type = "PROCEDURAL"

    importance = str(
        claim.get("importance", "")
    ).upper()
    if importance not in IMPORTANCE_VALUES:
        importance = "SECONDARY"

    result = str(
        claim.get("result", "")
    ).upper()
    if result not in RESULT_VALUES:
        result = "UNCERTAIN"

    evidence_ids = [
        str(value)
        for value in claim.get(
            "case_evidence_ids",
            [],
        )
        if str(value) in case.valid_chunk_ids
    ]
    law_aids = []
    for value in claim.get("law_aids", []):
        try:
            aid = int(value)
        except (TypeError, ValueError):
            continue
        if aid in case.valid_law_aids:
            law_aids.append(aid)

    return {
        "claim_id": str(
            claim.get("claim_id", "")
        ),
        "claim_type": claim_type,
        "claim": truncate_words(
            claim.get("claim", ""),
            45,
        ),
        "importance": importance,
        "result": result,
        "case_evidence_ids": dedupe_preserve(
            evidence_ids
        ),
        "law_aids": dedupe_preserve(law_aids),
        "short_reason": truncate_words(
            claim.get("short_reason", ""),
            60,
        ),
    }


def parse_model_result(
    raw: dict[str, Any],
    case: CaseBundle,
    model_id: str,
) -> dict[str, Any]:
    prediction = str(
        raw.get("prediction", "")
    ).upper()
    if prediction not in LABELS:
        prediction = None

    probabilities = normalize_probabilities(
        raw.get("probabilities"),
        prediction,
    )
    if prediction is None:
        prediction = top_label(probabilities)

    claims_raw = raw.get("claims", [])
    if not isinstance(claims_raw, list):
        claims_raw = []
    claims = [
        validate_claim(item, case)
        for item in claims_raw
        if isinstance(item, dict)
    ][:12]

    confidence = str(
        raw.get("confidence", "LOW")
    ).upper()
    if confidence not in {
        "HIGH",
        "MEDIUM",
        "LOW",
    }:
        confidence = "LOW"

    return {
        "case_id": case.case_id,
        "model_id": model_id,
        "prediction": prediction,
        "probabilities": probabilities,
        "claims": claims,
        "confidence": confidence,
        "decision_summary": truncate_words(
            raw.get("decision_summary", ""),
            130,
        ),
    }


def run_model_pass(
    cases: Sequence[CaseBundle],
    runtime: Any,
    output_dir: Path,
    *,
    system_prompt: str,
    max_new_tokens: int,
    pass_name: str,
) -> dict[str, dict[str, Any]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    results = {}

    for position, case in enumerate(cases, start=1):
        path = output_dir / f"{case.case_id}.json"
        if path.exists():
            results[case.case_id] = read_json(path)
            continue

        user = f"""
ĐỊNH NGHĨA NHÃN:
{label_definition_text()}

CASE:
{case.packet_text}

Hãy phân rã claims rồi dự đoán.
{prediction_schema()}
"""
        print(
            f"[{pass_name} {position}/{len(cases)}] "
            f"{case.case_id}"
        )
        try:
            raw = runtime.generate_json(
                system_prompt,
                user,
                max_new_tokens,
            )
            payload = parse_model_result(
                raw,
                case,
                getattr(
                    runtime,
                    "model_id",
                    getattr(
                        runtime.config,
                        "primary_model_id",
                        pass_name,
                    ),
                ),
            )
            payload["pass_name"] = pass_name
        except Exception as exc:
            payload = {
                "case_id": case.case_id,
                "model_id": getattr(
                    runtime,
                    "model_id",
                    pass_name,
                ),
                "pass_name": pass_name,
                "prediction": "PARTIAL_A_WIN",
                "probabilities": {
                    "A_WIN": 0.20,
                    "PARTIAL_A_WIN": 0.35,
                    "PARTIAL_B_WIN": 0.25,
                    "B_WIN": 0.20,
                },
                "claims": [],
                "confidence": "LOW",
                "decision_summary": "",
                "error": (
                    f"{type(exc).__name__}: {exc}"
                ),
            }

        atomic_write_json(path, payload)
        results[case.case_id] = payload

    return results


# =============================================================================
# Deterministic claim adjudication and consensus
# =============================================================================

def claim_table_probabilities(
    claims: Sequence[dict[str, Any]],
) -> dict[str, float]:
    a_points = 0.0
    b_points = 0.0
    total_weight = 0.0

    a_core_results = []
    b_core_results = []

    for claim in claims:
        claim_type = claim.get("claim_type")
        importance = claim.get("importance", "SECONDARY")
        result = claim.get("result", "UNCERTAIN")

        weight = IMPORTANCE_WEIGHT.get(
            importance,
            1.0,
        )
        if claim_type == "PROCEDURAL":
            weight *= 0.55

        fraction = GRANT_FRACTION.get(
            result,
            0.5,
        )

        if claim_type == "A_CLAIM":
            a_points += weight * fraction
            b_points += weight * (1.0 - fraction)
            if importance == "CORE":
                a_core_results.append(fraction)
        elif claim_type in {
            "B_COUNTERCLAIM",
            "B_DEFENSE",
        }:
            b_points += weight * fraction
            a_points += weight * (1.0 - fraction)
            if importance == "CORE":
                b_core_results.append(fraction)
        else:
            a_points += weight * 0.5
            b_points += weight * 0.5

        total_weight += weight

    if total_weight <= 0:
        return {
            label: 0.25
            for label in LABELS
        }

    a_share = a_points / (a_points + b_points)
    a_core = (
        float(np.mean(a_core_results))
        if a_core_results else a_share
    )
    b_core = (
        float(np.mean(b_core_results))
        if b_core_results else 1.0 - a_share
    )

    # Smooth rule scores instead of one hard label.
    scores = {
        "A_WIN": max(
            0.0,
            1.0 - abs(a_share - 0.92) / 0.42,
        ),
        "PARTIAL_A_WIN": max(
            0.0,
            1.0 - abs(a_share - 0.66) / 0.32,
        ),
        "PARTIAL_B_WIN": max(
            0.0,
            1.0 - abs(a_share - 0.40) / 0.28,
        ),
        "B_WIN": max(
            0.0,
            1.0 - abs(a_share - 0.08) / 0.38,
        ),
    }

    # Core-claim corrections.
    if a_core >= 0.90 and b_core <= 0.25:
        scores["A_WIN"] += 0.45
    elif 0.50 <= a_core < 0.90:
        scores["PARTIAL_A_WIN"] += 0.30

    if a_core <= 0.10 and b_core >= 0.75:
        scores["B_WIN"] += 0.45
    elif a_core < 0.50 and b_core >= 0.50:
        scores["PARTIAL_B_WIN"] += 0.30

    values = np.asarray(
        [max(1e-6, scores[label]) for label in LABELS],
        dtype=np.float64,
    )
    values /= values.sum()
    return {
        label: float(values[index])
        for index, label in enumerate(LABELS)
    }


def result_reliability(
    result: dict[str, Any],
    config: NoPublicOutcomeConfig,
) -> float:
    claims = result.get("claims", [])
    if not isinstance(claims, list):
        claims = []

    claim_factor = min(
        1.0,
        len(claims)
        / max(
            1,
            config.min_valid_claims_for_full_weight,
        ),
    )

    evidence_claims = sum(
        bool(claim.get("case_evidence_ids"))
        for claim in claims
    )
    law_claims = sum(
        bool(claim.get("law_aids"))
        for claim in claims
    )
    denominator = max(1, len(claims))

    evidence_ratio = evidence_claims / denominator
    law_ratio = law_claims / denominator

    confidence_factor = {
        "HIGH": 1.0,
        "MEDIUM": 0.82,
        "LOW": 0.62,
    }.get(result.get("confidence"), 0.62)

    return min(
        1.15,
        0.55
        + 0.22 * claim_factor
        + config.evidence_citation_bonus * evidence_ratio
        + config.law_citation_bonus * law_ratio
        + 0.20 * confidence_factor,
    )


def consensus_probabilities(
    primary: dict[str, Any],
    auditor: dict[str, Any],
    config: NoPublicOutcomeConfig,
) -> dict[str, float]:
    primary_direct = normalize_probabilities(
        primary.get("probabilities"),
        primary.get("prediction"),
    )
    auditor_direct = normalize_probabilities(
        auditor.get("probabilities"),
        auditor.get("prediction"),
    )
    primary_claim = claim_table_probabilities(
        primary.get("claims", [])
    )
    auditor_claim = claim_table_probabilities(
        auditor.get("claims", [])
    )

    primary_reliability = result_reliability(
        primary,
        config,
    )
    auditor_reliability = result_reliability(
        auditor,
        config,
    )

    weights = {
        "primary_direct": (
            config.direct_primary_weight
            * primary_reliability
        ),
        "primary_claim": (
            config.claim_primary_weight
            * primary_reliability
        ),
        "auditor_direct": (
            config.direct_auditor_weight
            * auditor_reliability
        ),
        "auditor_claim": (
            config.claim_auditor_weight
            * auditor_reliability
        ),
    }

    scores = {}
    for label in LABELS:
        scores[label] = (
            weights["primary_direct"]
            * primary_direct[label]
            + weights["primary_claim"]
            * primary_claim[label]
            + weights["auditor_direct"]
            * auditor_direct[label]
            + weights["auditor_claim"]
            * auditor_claim[label]
        )

    if primary.get("prediction") != auditor.get("prediction"):
        joint_claim = {
            label: (
                0.5 * primary_claim[label]
                + 0.5 * auditor_claim[label]
            )
            for label in LABELS
        }
        rule_label = top_label(joint_claim)
        scores[rule_label] += (
            config.disagreement_claim_rule_bonus
        )

    total = sum(scores.values())
    if total <= 0:
        return {label: 0.25 for label in LABELS}
    return {
        label: float(scores[label] / total)
        for label in LABELS
    }


# =============================================================================
# Submission
# =============================================================================

def build_law_evidence_objects(
    aids: Sequence[int],
    article_by_aid: dict[int, LawArticle],
) -> list[dict[str, Any]]:
    output = []
    seen = set()

    for value in aids:
        aid = int(value)
        article = article_by_aid.get(aid)
        if article is None:
            raise ValueError(
                f"aid {aid} không có trong corpus."
            )
        key = (article.law_id, aid)
        if key in seen:
            continue
        seen.add(key)
        output.append({
            "law_id": article.law_id,
            "aid": aid,
        })
    return output


def validate_submission(
    rows: Sequence[dict[str, Any]],
    original_private_cases: Sequence[dict[str, Any]],
    article_by_aid: dict[int, LawArticle],
) -> dict[str, Any]:
    expected_ids = [
        str(item["case_id"])
        for item in original_private_cases
    ]
    actual_ids = [
        str(item.get("case_id"))
        for item in rows
    ]
    errors = []

    if actual_ids != expected_ids:
        errors.append(
            "Tập/thứ tự case_id không khớp private input."
        )
    if len(actual_ids) != len(set(actual_ids)):
        errors.append("Trùng case_id.")

    label_counts = Counter()
    case_counts = []
    law_counts = []

    for row in rows:
        case_id = row.get("case_id")
        prediction = row.get("prediction")
        if prediction not in LABELS:
            errors.append(
                f"{case_id}: prediction không hợp lệ."
            )
        else:
            label_counts[prediction] += 1

        case_evidence = row.get("case_evidence")
        if not isinstance(case_evidence, list):
            errors.append(
                f"{case_id}: case_evidence không phải list."
            )
            case_evidence = []
        if len(case_evidence) != len(
            set(map(str, case_evidence))
        ):
            errors.append(
                f"{case_id}: trùng case_evidence."
            )
        case_counts.append(len(case_evidence))

        law_evidence = row.get("law_evidence")
        if not isinstance(law_evidence, list):
            errors.append(
                f"{case_id}: law_evidence không phải list."
            )
            law_evidence = []

        pairs = []
        for item in law_evidence:
            if not isinstance(item, dict):
                errors.append(
                    f"{case_id}: law item không phải object."
                )
                continue
            if set(item) != {"law_id", "aid"}:
                errors.append(
                    f"{case_id}: law item phải có đúng law_id, aid."
                )
            try:
                aid = int(item.get("aid"))
            except (TypeError, ValueError):
                errors.append(
                    f"{case_id}: aid không phải số nguyên."
                )
                continue
            article = article_by_aid.get(aid)
            if article is None:
                errors.append(
                    f"{case_id}: aid {aid} không tồn tại."
                )
                continue
            if item.get("law_id") != article.law_id:
                errors.append(
                    f"{case_id}: law_id không khớp aid {aid}."
                )
            pairs.append((item.get("law_id"), aid))

        if len(pairs) != len(set(pairs)):
            errors.append(
                f"{case_id}: trùng law evidence."
            )
        law_counts.append(len(law_evidence))

    return {
        "valid": not errors,
        "errors": errors,
        "case_count": len(rows),
        "label_distribution": dict(label_counts),
        "mean_case_evidence": (
            float(np.mean(case_counts))
            if case_counts else 0.0
        ),
        "mean_law_evidence": (
            float(np.mean(law_counts))
            if law_counts else 0.0
        ),
    }


def build_submission(
    cases: Sequence[CaseBundle],
    original_private_cases: Sequence[dict[str, Any]],
    primary_results: dict[str, dict[str, Any]],
    auditor_results: dict[str, dict[str, Any]],
    article_by_aid: dict[int, LawArticle],
    output_dir: Path,
    config: NoPublicOutcomeConfig,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    case_by_id = {case.case_id: case for case in cases}

    rows = []
    diagnostics = []

    for original in original_private_cases:
        case_id = str(original["case_id"])
        case = case_by_id[case_id]
        primary = primary_results[case_id]
        auditor = auditor_results[case_id]
        probabilities = consensus_probabilities(
            primary,
            auditor,
            config,
        )
        prediction = top_label(probabilities)

        row = {
            "case_id": case_id,
            "prediction": prediction,
            "case_evidence": dedupe_preserve([
                str(value)
                for value in case.selected_chunk_ids
                if str(value)
            ]),
            "law_evidence": build_law_evidence_objects(
                case.selected_law_aids,
                article_by_aid,
            ),
        }
        rows.append(row)

        diagnostics.append({
            "case_id": case_id,
            "prediction": prediction,
            "probabilities": probabilities,
            "margin": probability_margin(probabilities),
            "primary_prediction": primary["prediction"],
            "auditor_prediction": auditor["prediction"],
            "primary_claim_rule": top_label(
                claim_table_probabilities(
                    primary.get("claims", [])
                )
            ),
            "auditor_claim_rule": top_label(
                claim_table_probabilities(
                    auditor.get("claims", [])
                )
            ),
            "primary_reliability": result_reliability(
                primary,
                config,
            ),
            "auditor_reliability": result_reliability(
                auditor,
                config,
            ),
            "case_evidence_count": len(
                row["case_evidence"]
            ),
            "law_evidence_count": len(
                row["law_evidence"]
            ),
        })

    if (
        config.expected_private_cases
        and len(rows) != config.expected_private_cases
    ):
        raise AssertionError(
            f"Expected {config.expected_private_cases} cases, "
            f"nhận {len(rows)}."
        )

    atomic_write_json(
        output_dir / "submission.json",
        rows,
    )
    atomic_write_json(
        output_dir / "submission_diagnostics.json",
        diagnostics,
    )

    validation = validate_submission(
        rows,
        original_private_cases,
        article_by_aid,
    )
    atomic_write_json(
        output_dir / "submission_validation.json",
        validation,
    )
    if not validation["valid"]:
        raise RuntimeError(
            "Submission không hợp lệ: "
            + "; ".join(validation["errors"])
        )

    return rows, diagnostics


# =============================================================================
# Complete no-public pipeline
# =============================================================================

def run_no_public_pipeline(
    *,
    private_test_path: Path,
    law_corpus_path: Path,
    case_retrieval_dir: Path,
    private_case_and_law_path: Path,
    work_dir: Path,
    config: NoPublicOutcomeConfig,
) -> dict[str, Any]:
    work_dir.mkdir(parents=True, exist_ok=True)
    atomic_write_json(
        work_dir / "config.json",
        asdict(config),
    )

    article_by_aid = load_law_corpus(
        law_corpus_path
    )
    cases, original_private_cases = (
        load_private_cases(
            private_test_path,
            case_retrieval_dir,
            private_case_and_law_path,
        )
    )

    for case in cases:
        build_case_packet(
            case,
            article_by_aid,
            config,
        )

    atomic_write_json(
        work_dir / "case_manifest.json",
        [
            {
                "case_id": case.case_id,
                "case_evidence_count": len(
                    case.selected_chunk_ids
                ),
                "law_evidence_count": len(
                    case.selected_law_aids
                ),
                "packet_words": len(
                    case.packet_text.split()
                ),
            }
            for case in cases
        ],
    )

    primary_runtime = QwenPrimaryRuntime(config)
    primary_runtime.load()
    primary_results = run_model_pass(
        cases,
        primary_runtime,
        work_dir / "primary_predictions",
        system_prompt=PRIMARY_SYSTEM,
        max_new_tokens=config.primary_max_new_tokens,
        pass_name="primary_qwen",
    )
    primary_runtime.unload()

    auditor_runtime = TextAuditorRuntime(config)
    auditor_runtime.load()
    auditor_results = run_model_pass(
        cases,
        auditor_runtime,
        work_dir / "auditor_predictions",
        system_prompt=AUDITOR_SYSTEM,
        max_new_tokens=config.auditor_max_new_tokens,
        pass_name="independent_auditor",
    )
    auditor_model_used = auditor_runtime.model_id
    auditor_fallback_used = (
        auditor_runtime.using_fallback
    )
    auditor_runtime.unload()

    rows, diagnostics = build_submission(
        cases,
        original_private_cases,
        primary_results,
        auditor_results,
        article_by_aid,
        work_dir / "final",
        config,
    )

    summary = {
        "public_data_used": False,
        "private_case_count": len(cases),
        "primary_model": config.primary_model_id,
        "auditor_model_requested": (
            config.auditor_model_id
        ),
        "auditor_model_used": auditor_model_used,
        "auditor_fallback_used": auditor_fallback_used,
        "prediction_distribution": dict(
            Counter(
                row["prediction"]
                for row in rows
            )
        ),
        "primary_auditor_agreement": float(
            np.mean([
                primary_results[case.case_id]["prediction"]
                == auditor_results[case.case_id]["prediction"]
                for case in cases
            ])
        ),
        "mean_final_margin": float(
            np.mean([
                row["margin"]
                for row in diagnostics
            ])
        ),
        "submission_path": str(
            work_dir / "final" / "submission.json"
        ),
    }
    atomic_write_json(
        work_dir / "run_summary.json",
        summary,
    )
    return summary


Overwriting /content/alqac_outcome_no_public.py


## 2. Cấu trúc thư mục trên Colab Runtime (/content)

Khi bạn upload folder `data` trực tiếp lên Colab runtime (thư mục `/content`):
```text
/content/
└── data/
    ├── ground_truth.json
    └── public_test/
        ├── ALQAC_public_test.json
        ├── corpus_law_pub.json
        ├── public_case_and_law_evidence.json
        └── case_retrieval_outputs/
            ├── case_1087_evidence_result.json
            └── ... (50 cases)
```
Tất cả kết quả chạy và submission sẽ được lưu trực tiếp vào thư mục runtime `/content/submissions/`.


In [8]:
sys.path.insert(0, "/content")

from alqac_outcome_no_public import (
    NoPublicOutcomeConfig,
    read_json,
    run_no_public_pipeline,
)

# =============================================================================
# CẤU HÌNH ĐƯỜNG DẪN TRỰC TIẾP TRÊN COLAB RUNTIME (/content)
# =============================================================================
POSSIBLE_DATA_DIRS = [
    Path("/content/data"),
    Path("/content/data/public_test"),
    Path("/content/public_test"),
    Path("/content"),
    Path("./data"),
    Path("."),
]

DATA_DIR = None
DATA_ROOT = None

for p in POSSIBLE_DATA_DIRS:
    if not p.exists():
        continue
    # Trường hợp 1: Có thư mục public_test bên trong (data/public_test)
    if (p / "public_test" / "ALQAC_public_test.json").exists():
        DATA_ROOT = p
        DATA_DIR = p / "public_test"
        break
    # Trường hợp 2: Các file nằm trực tiếp trong p
    elif (p / "ALQAC_public_test.json").exists():
        DATA_ROOT = p.parent if p.name == "public_test" else p
        DATA_DIR = p
        break

if DATA_DIR is None:
    DATA_ROOT = Path("/content/data")
    DATA_DIR = DATA_ROOT / "public_test" if (DATA_ROOT / "public_test").exists() else DATA_ROOT

# ROOT lưu output và submissions trên Colab runtime
ROOT = Path("/content")

# Thư mục lưu kết quả submission trên Colab runtime
SUBMISSION_DIR = ROOT / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

PRIVATE_TEST_PATH = DATA_DIR / "ALQAC_public_test.json"
LAW_CORPUS_PATH = DATA_DIR / "corpus_law_pub.json"
CASE_RETRIEVAL_DIR = DATA_DIR / "case_retrieval_outputs"
PRIVATE_CASE_AND_LAW_PATH = DATA_DIR / "public_case_and_law_evidence.json"

# Tìm ground_truth.json
GROUND_TRUTH_PATH = None
for gt_candidate in [
    DATA_ROOT / "ground_truth.json",
    DATA_DIR / "ground_truth.json",
    Path("/content/data/ground_truth.json"),
    Path("/content/ground_truth.json"),
]:
    if gt_candidate.exists():
        GROUND_TRUTH_PATH = gt_candidate
        break

RUN_NAME = "outcome_public_eval_primary"
WORK_DIR = ROOT / "outcome_prediction_runs" / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)

for path in [
    PRIVATE_TEST_PATH,
    LAW_CORPUS_PATH,
    CASE_RETRIEVAL_DIR,
    PRIVATE_CASE_AND_LAW_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")

print("==================================================================")
print("🚀 ĐÃ KẾT NỐI THƯ MỤC DATA TRÊN RUNTIME THÀNH CÔNG:")
print("   - ROOT thư mục làm việc :", ROOT)
print("   - Thư mục DATA          :", DATA_DIR)
print("   - Thư mục SUBMISSIONS   :", SUBMISSION_DIR)
print("   - File GROUND TRUTH     :", GROUND_TRUTH_PATH)
print("   - File PRIVATE TEST     :", PRIVATE_TEST_PATH)
print("   - Thư mục CHUNKS FACT   :", CASE_RETRIEVAL_DIR)
print("==================================================================")


🚀 ĐÃ KẾT NỐI THƯ MỤC DATA TRÊN RUNTIME THÀNH CÔNG:
   - ROOT thư mục làm việc : /content
   - Thư mục DATA          : /content/data/public_test
   - Thư mục SUBMISSIONS   : /content/submissions
   - File GROUND TRUTH     : /content/data/ground_truth.json
   - File PRIVATE TEST     : /content/data/public_test/ALQAC_public_test.json
   - Thư mục CHUNKS FACT   : /content/data/public_test/case_retrieval_outputs


## 3. Hugging Face token và cấu hình

Token không được ghi vào file output.

Để tắt fallback và bắt buộc Aya:

```python
allow_auditor_fallback=False
```


In [9]:
HF_TOKEN = os.getenv("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass(
        "Hugging Face token cho Aya (Enter để dùng fallback): "
    ).strip()

CONFIG = NoPublicOutcomeConfig(
    primary_model_id="Qwen/Qwen3.5-9B",
    auditor_model_id="CohereLabs/aya-expanse-8b",
    auditor_fallback_model_id="Qwen/Qwen3-8B",

    primary_quant_mode="bf16",
    auditor_quant_mode="bf16",
    allow_auditor_fallback=True,
    hf_token=HF_TOKEN or None,

    deterministic_generation=True,
    qwen_enable_thinking=False,

    max_chunks=24,
    max_total_evidence_words=1900,
    max_total_law_words=900,

    expected_private_cases=50,
)

safe_config = {
    key: value
    for key, value in CONFIG.__dict__.items()
    if key != "hf_token"
}
print(json.dumps(
    safe_config,
    ensure_ascii=False,
    indent=2,
))


Hugging Face token cho Aya (Enter để dùng fallback): ··········
{
  "primary_model_id": "Qwen/Qwen3.5-9B",
  "auditor_model_id": "CohereLabs/aya-expanse-8b",
  "auditor_fallback_model_id": "Qwen/Qwen3-8B",
  "primary_quant_mode": "bf16",
  "auditor_quant_mode": "bf16",
  "allow_auditor_fallback": true,
  "deterministic_generation": true,
  "qwen_enable_thinking": false,
  "max_chunks": 24,
  "max_words_per_chunk": 125,
  "max_total_evidence_words": 1900,
  "max_law_articles": 16,
  "max_words_per_law": 150,
  "max_total_law_words": 900,
  "max_hint_words": 230,
  "max_outcome_clues": 10,
  "primary_max_new_tokens": 950,
  "auditor_max_new_tokens": 950,
  "direct_primary_weight": 0.32,
  "claim_primary_weight": 0.22,
  "direct_auditor_weight": 0.27,
  "claim_auditor_weight": 0.19,
  "min_valid_claims_for_full_weight": 2,
  "evidence_citation_bonus": 0.08,
  "law_citation_bonus": 0.05,
  "disagreement_claim_rule_bonus": 0.08,
  "expected_private_cases": 50
}


## 4. Chạy pipeline

Quy trình chạy tuần tự để tiết kiệm VRAM:

```text
load Qwen → primary 60 case → unload
load Aya → audit 60 case → unload
build deterministic consensus → submission
```

Chạy lại cell sau khi Colab ngắt sẽ bỏ qua các case đã có JSON.


In [10]:
summary = run_no_public_pipeline(
    private_test_path=PRIVATE_TEST_PATH,
    law_corpus_path=LAW_CORPUS_PATH,
    case_retrieval_dir=CASE_RETRIEVAL_DIR,
    private_case_and_law_path=PRIVATE_CASE_AND_LAW_PATH,
    work_dir=WORK_DIR,
    config=CONFIG,
)

print(json.dumps(
    summary,
    ensure_ascii=False,
    indent=2,
))


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded auditor: Qwen/Qwen3-8B fallback= True
[independent_auditor 1/50] case_4101
[independent_auditor 2/50] case_4337
[independent_auditor 3/50] case_4588
[independent_auditor 4/50] case_4616
[independent_auditor 5/50] case_6616
[independent_auditor 6/50] case_5226
[independent_auditor 7/50] case_6284
[independent_auditor 8/50] case_3995
[independent_auditor 9/50] case_6384
[independent_auditor 10/50] case_8219
[independent_auditor 11/50] case_2705
[independent_auditor 12/50] case_3263
[independent_auditor 13/50] case_9125
[independent_auditor 14/50] case_5658
[independent_auditor 15/50] case_5045
[independent_auditor 16/50] case_4920
[independent_auditor 17/50] case_8758
[independent_auditor 18/50] case_9089
[independent_auditor 19/50] case_127
[independent_auditor 20/50] case_2978
[independent_auditor 21/50] case_4579
[independent_auditor 22/50] case_5236
[independent_auditor 23/50] case_6663
[independent_auditor 24/50] case_8732
[independent_auditor 25/50] case_4861
[independent_au

## 5. Kiểm tra submission

Không có public accuracy để báo cáo. Thay vào đó kiểm tra:

- primary/auditor agreement;
- final probability margin;
- các case bất đồng;
- định dạng submission;
- `law_id` có khớp `aid` hay không.


In [11]:
SUBMISSION_PATH = (
    WORK_DIR / "final" / "submission.json"
)
VALIDATION_PATH = (
    WORK_DIR
    / "final"
    / "submission_validation.json"
)
DIAGNOSTICS_PATH = (
    WORK_DIR
    / "final"
    / "submission_diagnostics.json"
)

submission = read_json(SUBMISSION_PATH)
validation = read_json(VALIDATION_PATH)
diagnostics = read_json(DIAGNOSTICS_PATH)

print(json.dumps(
    validation,
    ensure_ascii=False,
    indent=2,
))

if not validation["valid"]:
    raise RuntimeError(validation["errors"])

diagnostic_df = pd.DataFrame(diagnostics)
display(
    diagnostic_df.sort_values("margin").head(25)
)


{
  "valid": true,
  "errors": [],
  "case_count": 50,
  "label_distribution": {
    "PARTIAL_A_WIN": 37,
    "A_WIN": 6,
    "B_WIN": 7
  },
  "mean_case_evidence": 35.32,
  "mean_law_evidence": 39.52
}


,case_id,prediction,probabilities,margin,primary_prediction,auditor_prediction,primary_claim_rule,auditor_claim_rule,primary_reliability,auditor_reliability,case_evidence_count,law_evidence_count
9,case_8219,A_WIN,"{'A_WIN': 0.3175079575596817, 'PARTIAL_A_WIN':...",0.069898,A_WIN,PARTIAL_A_WIN,A_WIN,A_WIN,0.674000,0.674,13,18
18,case_127,A_WIN,"{'A_WIN': 0.3175079575596817, 'PARTIAL_A_WIN':...",0.069898,A_WIN,PARTIAL_A_WIN,A_WIN,A_WIN,0.674000,0.674,48,44
45,case_2035,A_WIN,"{'A_WIN': 0.3175079575596817, 'PARTIAL_A_WIN':...",0.069898,A_WIN,PARTIAL_A_WIN,A_WIN,A_WIN,0.674000,0.674,12,48
43,case_4557,B_WIN,"{'A_WIN': 0.1052916299577857, 'PARTIAL_A_WIN':...",0.139121,PARTIAL_B_WIN,PARTIAL_A_WIN,B_WIN,A_WIN,1.100000,0.674,14,25
42,case_669,B_WIN,"{'A_WIN': 0.10571653128146552, 'PARTIAL_A_WIN'...",0.139379,PARTIAL_B_WIN,PARTIAL_A_WIN,B_WIN,A_WIN,1.083333,0.674,11,35
15,case_4920,A_WIN,"{'A_WIN': 0.492181409992655, 'PARTIAL_A_WIN': ...",0.142481,A_WIN,PARTIAL_A_WIN,PARTIAL_A_WIN,A_WIN,1.066667,0.674,13,35
30,case_2238,PARTIAL_A_WIN,"{'A_WIN': 0.15389758106854742, 'PARTIAL_A_WIN'...",0.213571,PARTIAL_A_WIN,PARTIAL_A_WIN,PARTIAL_B_WIN,A_WIN,1.083333,0.674,214,30
3,case_4616,PARTIAL_A_WIN,"{'A_WIN': 0.3016227012055108, 'PARTIAL_A_WIN':...",0.214738,PARTIAL_A_WIN,PARTIAL_A_WIN,A_WIN,A_WIN,1.087500,0.674,81,48
28,case_5160,PARTIAL_A_WIN,"{'A_WIN': 0.11670205862078654, 'PARTIAL_A_WIN'...",0.217145,PARTIAL_A_WIN,PARTIAL_A_WIN,PARTIAL_B_WIN,A_WIN,1.039000,0.674,3,37
2,case_4588,PARTIAL_A_WIN,"{'A_WIN': 0.11516561431304559, 'PARTIAL_A_WIN'...",0.239863,PARTIAL_A_WIN,PARTIAL_A_WIN,PARTIAL_B_WIN,A_WIN,1.083333,0.674,11,46


## 6. Sao chép file nộp bài

Tạo:

```text
MyDrive/ALQAC/submission.json
```


In [12]:
# =============================================================================
# 6. LƯU TỰ ĐỘNG CÁC FILE SUBMISSION VÀO COLAB RUNTIME
# =============================================================================
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# 1. File submission thuần túy của Primary Model (Qwen3.5-9B)
primary_submission_path = SUBMISSION_DIR / "submission_primary.json"
primary_rows = [
    {
        "case_id": sub_row["case_id"],
        "prediction": diag["primary_prediction"],
        "case_evidence": sub_row["case_evidence"],
        "law_evidence": sub_row["law_evidence"],
    }
    for sub_row, diag in zip(submission, diagnostics)
]
with open(primary_submission_path, "w", encoding="utf-8") as f:
    json.dump(primary_rows, f, ensure_ascii=False, indent=2)
print("✅ [1/3] Đã lưu submission PRIMARY:", primary_submission_path)

# 2. File submission thuần túy của Auditor Model (Aya Expanse 8B / Qwen Fallback)
auditor_submission_path = SUBMISSION_DIR / "submission_auditor.json"
auditor_rows = [
    {
        "case_id": sub_row["case_id"],
        "prediction": diag["auditor_prediction"],
        "case_evidence": sub_row["case_evidence"],
        "law_evidence": sub_row["law_evidence"],
    }
    for sub_row, diag in zip(submission, diagnostics)
]
with open(auditor_submission_path, "w", encoding="utf-8") as f:
    json.dump(auditor_rows, f, ensure_ascii=False, indent=2)
print("✅ [2/3] Đã lưu submission AUDITOR:", auditor_submission_path)

# 3. File submission đồng thuận (Consensus giữa Primary và Auditor)
ensemble_consensus_path = SUBMISSION_DIR / "submission_ensemble_primary_auditor.json"
shutil.copy2(
    SUBMISSION_PATH,
    ensemble_consensus_path,
)
print("✅ [3/3] Đã lưu submission ENSEMBLE PRIMARY + AUDITOR:", ensemble_consensus_path)

# 4. Sao lưu chi tiết primary_predictions.json và auditor_predictions.json cho notebook ensemble tiếp theo
for pred_folder in ["primary_predictions", "auditor_predictions"]:
    src_folder = WORK_DIR / pred_folder
    if src_folder.exists():
        summary_preds = {}
        for p_file in src_folder.glob("*.json"):
            cid = p_file.stem
            try:
                summary_preds[cid] = read_json(p_file)
            except Exception:
                pass
        out_pred_file = SUBMISSION_DIR / f"{pred_folder}.json"
        with open(out_pred_file, "w", encoding="utf-8") as f:
            json.dump(summary_preds, f, ensure_ascii=False, indent=2)
        print(f"📦 Đã lưu tổng hợp {pred_folder}: {out_pred_file} ({len(summary_preds)} cases)")

# Nén thư mục submissions thành file zip trên runtime để tiện download
shutil.make_archive("/content/submissions_primary_auditor", "zip", SUBMISSION_DIR)
print(f"\n🎉 Tất cả 3 file submission đã được lưu tại: {SUBMISSION_DIR.resolve()}")
print("📦 Đã nén thành file zip để bạn tải về máy nếu muốn: /content/submissions_primary_auditor.zip")


✅ [1/3] Đã lưu submission PRIMARY: /content/submissions/submission_primary.json
✅ [2/3] Đã lưu submission AUDITOR: /content/submissions/submission_auditor.json
✅ [3/3] Đã lưu submission ENSEMBLE PRIMARY + AUDITOR: /content/submissions/submission_ensemble_primary_auditor.json
📦 Đã lưu tổng hợp primary_predictions: /content/submissions/primary_predictions.json (50 cases)
📦 Đã lưu tổng hợp auditor_predictions: /content/submissions/auditor_predictions.json (50 cases)

🎉 Tất cả 3 file submission đã được lưu tại: /content/submissions
📦 Đã nén thành file zip để bạn tải về máy nếu muốn: /content/submissions_primary_auditor.zip


## 7. Đánh giá Accuracy trên tập Public Test (Ground Truth)

In [13]:
if GROUND_TRUTH_PATH and GROUND_TRUTH_PATH.exists():
    print(f"\n=== ĐÁNH GIÁ ACCURACY TRÊN TẬP PUBLIC TEST ===")
    print(f"File Ground Truth: {GROUND_TRUTH_PATH}")
    gt_data = read_json(GROUND_TRUTH_PATH)
    gt_map = {item["case_id"]: item["verdict_label"] for item in gt_data}

    # Đọc kết quả từ các file submission vừa lưu
    primary_sub = read_json(primary_submission_path)
    auditor_sub = read_json(auditor_submission_path)
    ensemble_sub = read_json(ensemble_consensus_path)

    primary_pred_map = {row["case_id"]: row["prediction"] for row in primary_sub}
    auditor_pred_map = {row["case_id"]: row["prediction"] for row in auditor_sub}
    ensemble_pred_map = {row["case_id"]: row["prediction"] for row in ensemble_sub}

    correct_primary = 0
    correct_auditor = 0
    correct_ensemble = 0
    total = 0

    eval_rows = []
    for item in ensemble_sub:
        cid = item["case_id"]
        pred_p = primary_pred_map.get(cid, "N/A")
        pred_a = auditor_pred_map.get(cid, "N/A")
        pred_ens = ensemble_pred_map.get(cid, "N/A")

        gt = gt_map.get(cid)
        if gt:
            total += 1
            match_p = (pred_p == gt)
            match_a = (pred_a == gt)
            match_ens = (pred_ens == gt)

            if match_p: correct_primary += 1
            if match_a: correct_auditor += 1
            if match_ens: correct_ensemble += 1

            eval_rows.append({
                "case_id": cid,
                "ground_truth": gt,
                "primary": pred_p,
                "primary_ok": "✓" if match_p else "✗",
                "auditor": pred_a,
                "auditor_ok": "✓" if match_a else "✗",
                "ensemble_primary_auditor": pred_ens,
                "ensemble_ok": "✓" if match_ens else "✗",
            })

    acc_p = correct_primary / total if total > 0 else 0.0
    acc_a = correct_auditor / total if total > 0 else 0.0
    acc_ens = correct_ensemble / total if total > 0 else 0.0

    print("\n" + "=" * 70)
    print("🎯 BẢNG TỔNG HỢP ACCURACY TRÊN PUBLIC TEST (50 CASES)")
    print("=" * 70)
    print(f"  1. submission_primary.json                    : {correct_primary:2d}/{total} = {acc_p*100:.2f}%")
    print(f"  2. submission_auditor.json                    : {correct_auditor:2d}/{total} = {acc_a*100:.2f}%")
    print(f"  3. submission_ensemble_primary_auditor.json   : {correct_ensemble:2d}/{total} = {acc_ens*100:.2f}%")
    print("=" * 70 + "\n")

    eval_df = pd.DataFrame(eval_rows)
    print("Bảng so sánh chi tiết giữa Primary vs Auditor vs Ensemble (15 cases đầu):")
    display(eval_df.head(15))
else:
    print("Không tìm thấy ground_truth.json để đối chiếu Accuracy.")



=== ĐÁNH GIÁ ACCURACY TRÊN TẬP PUBLIC TEST ===
File Ground Truth: /content/data/ground_truth.json

🎯 BẢNG TỔNG HỢP ACCURACY TRÊN PUBLIC TEST (50 CASES)
  1. submission_primary.json                    : 20/50 = 40.00%
  2. submission_auditor.json                    : 19/50 = 38.00%
  3. submission_ensemble_primary_auditor.json   : 22/50 = 44.00%

Bảng so sánh chi tiết giữa Primary vs Auditor vs Ensemble (15 cases đầu):


,case_id,ground_truth,primary,primary_ok,auditor,auditor_ok,ensemble_primary_auditor,ensemble_ok
0,case_4101,PARTIAL_A_WIN,PARTIAL_A_WIN,✓,PARTIAL_A_WIN,✓,PARTIAL_A_WIN,✓
1,case_4337,A_WIN,A_WIN,✓,PARTIAL_A_WIN,✗,A_WIN,✓
2,case_4588,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
3,case_4616,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
4,case_6616,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
5,case_5226,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
6,case_6284,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
7,case_3995,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
8,case_6384,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
9,case_8219,A_WIN,A_WIN,✓,PARTIAL_A_WIN,✗,A_WIN,✓
